**Voice-to-Grasp - Experiment 4, combined analysis of the four participants (P1-P4)**: per-participant
accuracy, updates, loss and per-class results. Outputs were cleared for the release; the paper's
numbers and figures come from `Revision_ISR_2026/10_results_numbers.py` and `11_results_figures.py`.

# VoiceToGrasp — Multi-User Combined Analysis

Generates all combined figures across all participants for the paper.

**No manual paths needed** — just set the root folder and user names in Cell 1.  
The notebook automatically finds the latest timestamped subfolder for each user.  
Baseline accuracy is computed directly from the training CSV — no extra runs needed.

| Cell | Content |
|------|---------|
| 1 | Configuration — root folder + user names only |
| 2 | Load all users (auto-detects latest run per user) |
| 3 | Results table + Wilcoxon test (Table III in paper) |
| 4 | **Fig 1** — Learning curves, all users |
| 5 | **Fig 2** — Baseline vs Personalised grouped bar |
| 6 | **Fig 3** — Baseline vs Personalised box plot |
| 7 | **Fig 4** — Per-class F1 improvement heatmap |
| 8 | **Fig 5** — Per-class F1 grouped bar (mean across users) |
| 9 | **Fig 6** — Model updates per user + updates vs gain scatter |
| 10 | **Fig 7** — Aggregated confusion matrix (test phase) |
| 11 | **Fig 8** — Training loss curve per user |
| 12 | **Fig 9** — Confidence distribution correct vs incorrect |
| 13 | **Fig 10** — 6-panel summary dashboard |

## Cell 1 · Configuration

**Only edit this cell.**  Set `ROOT_DIR` to your `experiment_results` folder  
and list your user names in `USER_NAMES`.  Everything else is automatic.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)
from scipy.stats import wilcoxon

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.family":    "sans-serif",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.dpi":     120,
})

# ── EDIT THESE TWO LINES ONLY ─────────────────────────────────────────────
ROOT_DIR   = r"D:\PythonEnvs\nlpglove\Continual_Learning_Experiments\UserStudy_code_analysis\experiment_results"
USER_NAMES = ["P1", "P2", "P3", "P4"]  # list of all user IDs (e.g., "P3", "P4", "P2", "PilotA")
# ─────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = "./paper_figures"
SAVE_FIGS  = True    # False = display only, no files written
DPI        = 300     # publication quality

LABEL_ORDER = [
    "grasp", "open_hand", "gentle_grasp",
    "tight_grasp", "pinch_grasp", "tighten", "loosen",
]
LABEL_DISPLAY = {
    "grasp":        "Grasp",
    "open_hand":    "Open Hand",
    "gentle_grasp": "Gentle Grasp",
    "tight_grasp":  "Tight Grasp",
    "pinch_grasp":  "Pinch Grasp",
    "tighten":      "Tighten",
    "loosen":       "Loosen",
}
USER_COLORS = plt.cm.tab10.colors
os.makedirs(OUTPUT_DIR, exist_ok=True)

def _save(fig, name):
    if SAVE_FIGS:
        path = os.path.join(OUTPUT_DIR, name)
        fig.savefig(path, dpi=DPI, format="svg", bbox_inches="tight")
        print(f"  saved -> {path}")
    plt.show()

print("Config ready.")
print(f"Root: {ROOT_DIR}")
print(f"Users: {USER_NAMES}")


## Cell 2 · Load All Users

Automatically finds the **latest timestamped subfolder** inside each user's folder.  
Baseline accuracy is computed from the training CSV as the overall accuracy  
across all 140 adaptation samples — this reflects the model's performance  
before it has fully personalised to the user.

In [ ]:
def find_latest_run(root, user_name):
    """Return path to the most recent timestamped subfolder for a user."""
    user_dir = os.path.join(root, user_name)
    runs = sorted(glob.glob(os.path.join(user_dir, "*")))
    # keep only directories that look like timestamps (all digits / underscores)
    runs = [r for r in runs if os.path.isdir(r)]
    if not runs:
        raise FileNotFoundError(
            f"No subfolders found in: {user_dir}\n"
            "Run continual_batch_learning.py for this user first."
        )
    return runs[-1]   # sorted alphabetically = latest timestamp last


def load_user(root, name):
    run_dir  = find_latest_run(root, name)
    train_df = pd.read_csv(os.path.join(run_dir, "results_training_phase.csv"))
    test_df  = pd.read_csv(os.path.join(run_dir, "results_test_phase.csv"))

    # running accuracy over the 140 adaptation samples
    train_df["running_acc"] = train_df["correct"].expanding().mean()

    # ── Baseline accuracy ──────────────────────────────────────────────────
    # Derived from the training CSV: overall accuracy across all 140 samples.
    # This is a conservative estimate of the frozen model's performance —
    # it includes early correct predictions (before learning) and later ones
    # (after some adaptation), so it slightly underestimates the true frozen
    # baseline but requires no extra model run.
    baseline_acc = train_df["correct"].mean()

    # ── Per-class F1 ──────────────────────────────────────────────────────
    rpt_test = classification_report(
        test_df["true_label"], test_df["pred_label"],
        labels=LABEL_ORDER, zero_division=0, output_dict=True,
    )
    rpt_train = classification_report(
        train_df["true_label"], train_df["pred_label"],
        labels=LABEL_ORDER, zero_division=0, output_dict=True,
    )

    return {
        "name":         name,
        "run_dir":      run_dir,
        "train_df":     train_df,
        "test_df":      test_df,
        "train_acc":    train_df["correct"].mean(),
        "test_acc":     test_df["correct"].mean(),
        "baseline_acc": baseline_acc,
        "updates":      int(train_df["loss"].notna().sum()),
        "f1_pers":      {l: rpt_test.get(l,  {}).get("f1-score", 0.0) for l in LABEL_ORDER},
        "f1_base":      {l: rpt_train.get(l, {}).get("f1-score", 0.0) for l in LABEL_ORDER},
    }


USERS = []
for name in USER_NAMES:
    try:
        u = load_user(ROOT_DIR, name)
        USERS.append(u)
        print(f"  {u['name']:12s}  run={os.path.basename(u['run_dir'])}  "
              f"baseline={u['baseline_acc']:.3f}  "
              f"test={u['test_acc']:.3f}  "
              f"gain={u['test_acc']-u['baseline_acc']:+.3f}  "
              f"updates={u['updates']}")
    except FileNotFoundError as e:
        print(f"  WARNING: skipping {name} — {e}")

print(f"\nLoaded {len(USERS)} user(s).")


## Cell 3 · Results Table + Wilcoxon Test

Prints **Table III** for the paper.  
Copy the numbers directly into your manuscript.

In [ ]:
fmt = "{:<14} {:>12} {:>12} {:>12} {:>11} {:>9} {:>12}"
print("=" * 86)
print("  RESULTS TABLE  (Table III)")
print("=" * 86)
print(fmt.format("Participant", "Baseline Acc", "Pers Test Acc",
                 "Gain (pp)", "Updates", "Mean Conf", "Train Acc"))
print("-" * 86)

gains, base_list, pers_list, upd_list = [], [], [], []
for u in USERS:
    base      = u["baseline_acc"]
    pers      = u["test_acc"]
    gain      = pers - base
    mean_conf = u["test_df"]["confidence"].mean()
    gains.append(gain); base_list.append(base)
    pers_list.append(pers); upd_list.append(u["updates"])
    print(fmt.format(
        u["name"],
        f"{base:.3f}", f"{pers:.3f}", f"{gain:+.3f}",
        str(u["updates"]), f"{mean_conf:.3f}", f"{u['train_acc']:.3f}"
    ))

print("-" * 86)
print(fmt.format(
    "Mean",
    f"{np.mean(base_list):.3f}", f"{np.mean(pers_list):.3f}",
    f"{np.mean(gains):+.3f}", f"{np.mean(upd_list):.1f}", "",
    f"{np.mean([u['train_acc'] for u in USERS]):.3f}"
))
print(fmt.format(
    "SD",
    f"{np.std(base_list):.3f}", f"{np.std(pers_list):.3f}",
    f"{np.std(gains):.3f}", f"{np.std(upd_list):.1f}", "", ""
))
print("=" * 86)

if len(USERS) >= 3:
    try:
        stat, pval = wilcoxon(pers_list, base_list, zero_method="zsplit")
        print(f"\nWilcoxon signed-rank test (personalised test acc vs baseline):")
        print(f"  W = {stat:.3f},  p = {pval:.4f}")
        print(f"  -> {'SIGNIFICANT' if pval < 0.05 else 'not significant'} at alpha = 0.05")
    except Exception as e:
        print(f"\nWilcoxon could not be computed: {e}")
else:
    print("\nNeed >= 3 users for Wilcoxon test.")


## Cell 4 · Fig 1 — Learning Curves (All Users)

Running accuracy over the 140 adaptation samples for every user on one plot,
with a dashed mean line. Use in the paper to show that all users improve.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for i, u in enumerate(USERS):
    acc = u["train_df"]["running_acc"].values
    ax.plot(range(1, len(acc) + 1), acc,
            color=USER_COLORS[i], linewidth=1.8, label=u["name"])
    ax.annotate(f"{acc[-1]:.2f}",
                xy=(len(acc), acc[-1]), xytext=(5, 0),
                textcoords="offset points",
                color=USER_COLORS[i], fontsize=8, va="center")

min_len  = min(len(u["train_df"]) for u in USERS)
mean_acc = np.array([u["train_df"]["running_acc"].values[:min_len]
                     for u in USERS]).mean(axis=0)
ax.plot(range(1, min_len + 1), mean_acc,
        "k--", linewidth=2.5, label="Mean (all users)", zorder=10)

ax.set_xlabel("Sample Number (Adaptation Phase)")
ax.set_ylabel("Cumulative Accuracy")
ax.set_title("Fig 1 — Running Accuracy During Continual Learning Adaptation\n"
             "(Phase 1, 140 samples per user)")
ax.set_ylim(0, 1.05); ax.set_xlim(1, 140)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
_save(fig, "fig1_learning_curves.svg")


## Cell 5 · Fig 2 — Baseline vs Personalised (Grouped Bar)

The **headline result figure**. For each user: baseline bar vs personalised bar,
with arrows and gain labels showing the improvement.

In [ ]:
names    = [u["name"]         for u in USERS]
baseline = [u["baseline_acc"] for u in USERS]
personal = [u["test_acc"]     for u in USERS]
gains    = [p - b for p, b in zip(personal, baseline)]
USER_LABELS = {
    "P2": "Participant_2",
    "P3":  "Participant_3",
    "P4": "Participant_4",
    "P1": "Participant_1",
    # "PilotA":    "Participant_5",
}
names = [USER_LABELS.get(n, n) for n in names]
x = np.arange(len(names)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))

bars_b = ax.bar(x - w/2, baseline, w, label="Baseline (training acc)",
                color="#5B9BD5", edgecolor="white")
bars_p = ax.bar(x + w/2, personal, w, label="Personalised (test acc)",
                color="#ED7D31", edgecolor="white")

for bar, val in zip(bars_b, baseline):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.2f}", ha="center", va="bottom",
            fontsize=9, color="#1F497D", fontweight="bold")
for bar, val in zip(bars_p, personal):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.2f}", ha="center", va="bottom",
            fontsize=9, color="#833C00", fontweight="bold")

for i, gain in enumerate(gains):
    color = "green" if gain >= 0 else "red"
    ax.annotate("", xy=(x[i] + w/2, personal[i]),
                xytext=(x[i] - w/2, baseline[i]),
                arrowprops=dict(arrowstyle="->", color=color, lw=1.4))
    ax.text(x[i], max(personal[i], baseline[i]) + 0.07,
            f"{gain:+.2f}", ha="center", fontsize=9,
            color=color, fontweight="bold")

ax.axhline(np.mean(baseline), color="#5B9BD5", linestyle=":", linewidth=1.2, alpha=0.7)
ax.axhline(np.mean(personal), color="#ED7D31", linestyle=":", linewidth=1.2, alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel("Accuracy")
ax.set_title("Baseline vs. Personalised Model Accuracy\n"
             "(arrows show gain per participant)")
ax.set_ylim(0, 1.2)
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
_save(fig, "fig2_personalised_vs_baseline_bars.svg")


## Cell 6 · Fig 3 — Box Plot Across Users

Statistical spread of accuracy across all participants. Pairs with the Wilcoxon test.

In [ ]:
baseline = [u["baseline_acc"] for u in USERS]
personal = [u["test_acc"]     for u in USERS]

fig, ax = plt.subplots(figsize=(6, 5))
bp = ax.boxplot(
    [baseline, personal],
    labels=["Baseline", "Personalised"],
    patch_artist=True, widths=0.45,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
)
bp["boxes"][0].set_facecolor("#5B9BD5")
bp["boxes"][1].set_facecolor("#ED7D31")

for i, (vals, color) in enumerate(
    zip([baseline, personal], ["#1F497D", "#833C00"]), start=1
):
    jitter = np.random.uniform(-0.07, 0.07, len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals,
               color=color, edgecolor="white", s=70, zorder=5)
    for j, (v, u) in enumerate(zip(vals, USERS)):
        ax.annotate(u["name"], xy=(i + jitter[j], v),
                    xytext=(6, 0), textcoords="offset points",
                    fontsize=7, va="center", color=color)

for i, vals in enumerate([baseline, personal], start=1):
    ax.text(i, np.mean(vals) + 0.04, f"μ = {np.mean(vals):.3f}",
            ha="center", fontsize=9, fontweight="bold")

ax.set_ylabel("Accuracy")
ax.set_title("Fig 3 — Accuracy Distribution Across All Participants")
ax.set_ylim(0, 1.15); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
_save(fig, "fig3_accuracy_boxplot.png")


## Cell 7 · Fig 4 — Per-Class F1 Improvement Heatmap

Rows = users, Columns = grasp classes.  
Value = F1(personalised test) − F1(training phase).  
Green = improved after personalisation, Red = degraded.

In [ ]:
rows = []
for u in USERS:
    row = {LABEL_DISPLAY[l]: round(u["f1_pers"][l] - u["f1_base"][l], 3)
           for l in LABEL_ORDER}
    rows.append(row)
names = [u["name"] for u in USERS]
USER_LABELS = {
    "P2": "Participant_2",
    "P3":  "Participant_3",
    "P4": "Participant_4",
    "P1": "Participant_1",
    # "PilotA":    "Participant_5",
}
names = [USER_LABELS.get(n, n) for n in names]
heat_df = pd.DataFrame(rows, index=names)
# heat_df = pd.DataFrame(rows, index=[u["name"] for u in USERS])


fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    heat_df, annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, vmin=-0.5, vmax=0.5,
    linewidths=0.5, linecolor="white", ax=ax,
    cbar_kws={"label": "F1 Change (Personalised − Baseline)"},
)
ax.set_yticklabels(names, rotation=20, ha="right")
ax.set_title("Fig 4 — Per-Class F1 Change After Personalisation\n"
             "(green = improvement, red = degradation)")
ax.set_xlabel("Grasp Category"); ax.set_ylabel("Participant")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
_save(fig, "fig4_f1_improvement_heatmap.svg")


## Cell 8 · Fig 5 — Per-Class F1 Grouped Bar (Mean Across Users)

For each grasp class: mean F1 baseline vs personalised across all users,
with SD error bars.

In [ ]:
mean_b = [np.mean([u["f1_base"][l] for u in USERS]) for l in LABEL_ORDER]
std_b  = [np.std( [u["f1_base"][l] for u in USERS]) for l in LABEL_ORDER]
mean_p = [np.mean([u["f1_pers"][l] for u in USERS]) for l in LABEL_ORDER]
std_p  = [np.std( [u["f1_pers"][l] for u in USERS]) for l in LABEL_ORDER]

x = np.arange(len(LABEL_ORDER)); w = 0.35
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(x - w/2, mean_b, w, yerr=std_b, capsize=4,
       label="Baseline", color="#5B9BD5", edgecolor="white",
       error_kw=dict(elinewidth=1.2, ecolor="grey", capthick=1.2))
ax.bar(x + w/2, mean_p, w, yerr=std_p, capsize=4,
       label="Personalised", color="#ED7D31", edgecolor="white",
       error_kw=dict(elinewidth=1.2, ecolor="grey", capthick=1.2))

ax.set_xticks(x)
ax.set_xticklabels([LABEL_DISPLAY[l] for l in LABEL_ORDER],
                   rotation=30, ha="right", fontsize=10)
ax.set_ylabel("F1-Score  (mean ± SD across participants)")
ax.set_ylim(0, 1.2)
ax.set_title("Per-Class F1: Baseline vs. Personalised\n"
             "(mean across all participants, error bars = SD)")
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
_save(fig, "fig5_per_class_f1_grouped.svg")


## Cell 9 · Fig 6 — Model Updates per User

Left: number of updates triggered per user (= misclassifications during adaptation).  
Right: scatter of updates vs accuracy gain.

In [ ]:
names   = [u["name"]    for u in USERS]
updates = [u["updates"] for u in USERS]
gains   = [u["test_acc"] - u["baseline_acc"] for u in USERS]
# names = [u["name"] for u in USERS]
USER_LABELS = {
    "P2": "Participant_2",
    "P3":  "Participant_3",
    "P4": "Participant_4",
    "P1": "Participant_1",
    # "PilotA":    "Participant_5",
}
names = [USER_LABELS.get(n, n) for n in names]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
bars = ax.bar(names, updates,
              color=[USER_COLORS[i] for i in range(len(USERS))],
              edgecolor="white", width=0.5)
for bar, val in zip(bars, updates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha="center", fontsize=11, fontweight="bold")
ax.axhline(np.mean(updates), color="black", linestyle="--",
           linewidth=1.2, label=f"Mean = {np.mean(updates):.1f}")
ax.set_ylabel("Number of Model Corrections (Phase 1)")
ax.set_title("Model Corrections Triggered per Participant")
ax.legend(fontsize=9); ax.set_ylim(0, max(updates) * 1.3)
ax.grid(axis="y", alpha=0.3)

ax = axes[1]
for i, u in enumerate(USERS):
    gain = u["test_acc"] - u["baseline_acc"]
    ax.scatter(u["updates"], gain, color=USER_COLORS[i], s=100, zorder=5)
    ax.annotate(names[i], xy=(u["updates"], gain),
                xytext=(5, 4), textcoords="offset points", fontsize=9)
if len(USERS) > 2:
    z  = np.polyfit(updates, gains, 1)
    xs = np.linspace(min(updates) - 2, max(updates) + 2, 100)
    ax.plot(xs, np.poly1d(z)(xs), "k--", linewidth=1, alpha=0.5, label="Trend")
ax.axhline(0, color="grey", linestyle=":", linewidth=0.8)
ax.set_xlabel("Number of Corrections in Phase 1")
ax.set_ylabel("Accuracy Gain  (Personalised − Baseline)")
ax.set_title("Corrections vs. Personalisation Gain")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Model Corrections Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
_save(fig, "fig6_updates_per_user.svg")


## Cell 10 · Fig 7 — Aggregated Confusion Matrix

All test-phase predictions from all users pooled into one matrix.

In [ ]:
all_true, all_pred = [], []
for u in USERS:
    all_true.extend(u["test_df"]["true_label"].tolist())
    all_pred.extend(u["test_df"]["pred_label"].tolist())

present = [l for l in LABEL_ORDER if l in set(all_true)]
cm      = confusion_matrix(all_true, all_pred, labels=present)

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(
    cm, display_labels=[LABEL_DISPLAY[l] for l in present]
).plot(ax=ax, colorbar=True, cmap="Blues", xticks_rotation=40)
ax.set_title(
    f"Aggregated Confusion Matrix — Personalised Model\n"
    f"Test Phase · {len(USERS)} participants · n = {len(all_true)} predictions"
)
plt.tight_layout()
_save(fig, "fig7_aggregated_confusion_matrix.svg")


## Cell 11 · Fig 8 — Training Loss Curve

Loss value at each model update during Phase 1, per user.
A declining trend confirms the model is learning from corrections.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
USER_LABELS = {
    "P2": "Participant_2",
    "P3":  "Participant_3",
    "P4": "Participant_4",
    "P1": "Participant_1",
    # "PilotA":    "Participant_5",
}

for i, u in enumerate(USERS):
    u["name"]= USER_LABELS.get(u["name"], u["name"])
    upd = u["train_df"].dropna(subset=["loss"]).copy()
    upd["loss"] = pd.to_numeric(upd["loss"], errors="coerce")
    if len(upd) == 0:
        continue
    ax.plot(upd["update_num"], upd["loss"],
            color=USER_COLORS[i], linewidth=1.5,
            marker="o", markersize=3, label=u["name"])
    roll = upd["loss"].rolling(5, min_periods=1).mean()
    ax.plot(upd["update_num"], roll,
            color=USER_COLORS[i], linewidth=2, linestyle="--", alpha=0.6)

ax.set_xlabel("Correction Number")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Training Loss at Each Model Correction (Phase 1)\n"
             "(solid = per-Correction, dashed = rolling mean over 5 updates)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
_save(fig, "fig8_loss_curve.svg")


# Statistical Analysis Cells — MultiUser_CombinedAnalysis_v3.ipynb
Add these cells **after Cell 3 (Results Table + Wilcoxon)** in your existing notebook.
They all use the `USERS` list already loaded in Cell 2.

---
## STAT CELL A · Effect Size (Cohen's d) — Baseline vs Personalised
Your existing Cell 3 already runs the Wilcoxon test. This cell adds
Cohen's d effect size, which reviewers increasingly require alongside p-values.

In [ ]:
import numpy as np
from scipy.stats import wilcoxon

baseline_accs = [u["baseline_acc"] for u in USERS]
personal_accs = [u["test_acc"]     for u in USERS]
gains         = [p - b for p, b in zip(personal_accs, baseline_accs)]

# Cohen's d for paired data
mean_diff = np.mean(gains)
std_diff  = np.std(gains, ddof=1)
cohen_d   = mean_diff / std_diff if std_diff > 0 else float('inf')

if   abs(cohen_d) >= 0.8: d_interp = "large"
elif abs(cohen_d) >= 0.5: d_interp = "medium"
elif abs(cohen_d) >= 0.2: d_interp = "small"
else:                     d_interp = "negligible"

# Wilcoxon (rerun cleanly for reporting)
stat_w, p_w = wilcoxon(personal_accs, baseline_accs, zero_method="zsplit")

print("=" * 60)
print("Personalisation Effect: Baseline vs Personalised Accuracy")
print("=" * 60)
print(f"  N participants          : {len(USERS)}")
print(f"  Mean baseline accuracy  : {np.mean(baseline_accs):.4f} (SD={np.std(baseline_accs):.4f})")
print(f"  Mean personalised acc   : {np.mean(personal_accs):.4f} (SD={np.std(personal_accs):.4f})")
print(f"  Mean gain               : {mean_diff:+.4f} (SD={std_diff:.4f})")
print(f"")
print(f"  Wilcoxon W              : {stat_w:.4f}")
print(f"  p-value                 : {p_w:.4f}")
sig = 'SIGNIFICANT' if p_w < 0.05 else 'not significant'
print(f"  -> {sig} at alpha = 0.05")
print(f"")
print(f"  Cohen's d (effect size) : {cohen_d:.4f} ({d_interp} effect)")
print("=" * 60)

---
## STAT CELL B · Bootstrap Confidence Intervals — Mean Accuracy Gain
Since your sample is small (n=5), bootstrap CIs are more reliable
than parametric CIs. This gives you a robust interval to report
alongside your Wilcoxon result.

In [ ]:
import numpy as np

np.random.seed(42)
gains = np.array([u["test_acc"] - u["baseline_acc"] for u in USERS])

n_boot = 10000
boot_means = [
    np.mean(np.random.choice(gains, size=len(gains), replace=True))
    for _ in range(n_boot)
]

ci_lo = np.percentile(boot_means, 2.5)
ci_hi = np.percentile(boot_means, 97.5)
observed_mean = np.mean(gains)

# One-sided p-value: proportion of bootstrap samples with gain <= 0
p_boot = np.mean(np.array(boot_means) <= 0)

print("=" * 60)
print(f"Bootstrap Analysis of Mean Accuracy Gain (n_boot={n_boot})")
print("=" * 60)
print(f"  Observed mean gain      : {observed_mean:+.4f}")
print(f"  95% Bootstrap CI        : [{ci_lo:+.4f}, {ci_hi:+.4f}]")
print(f"  Bootstrap p-value       : {p_boot:.4f} (one-sided: gain <= 0)")
sig = 'SIGNIFICANT' if p_boot < 0.05 else 'not significant'
print(f"  -> {sig} at alpha = 0.05")
print("")
print(f"  Per-user gains: {[round(g, 4) for g in gains]}")
print("=" * 60)

---
## STAT CELL C · Per-Class F1 Improvement — Wilcoxon Test Per Category
Tests whether the F1 improvement for each grasp category is statistically
significant across participants. Applies Bonferroni correction for
multiple comparisons (7 categories).

In [ ]:
from scipy.stats import wilcoxon
import numpy as np

alpha      = 0.05
n_tests    = len(LABEL_ORDER)
alpha_bonf = alpha / n_tests  # Bonferroni correction

print("=" * 75)
print("Per-Class F1 Improvement: Wilcoxon Test (with Bonferroni Correction)")
print(f"  Alpha = {alpha}, Bonferroni-corrected alpha = {alpha_bonf:.4f}")
print("=" * 75)

fmt = "{:<16} {:>10} {:>10} {:>10} {:>10} {:>12}"
print(fmt.format("Grasp Class", "Base F1", "Pers F1", "Mean Gain", "p-value", "Significant"))
print("-" * 75)

for label in LABEL_ORDER:
    base_f1s = [u["f1_base"][label] for u in USERS]
    pers_f1s = [u["f1_pers"][label] for u in USERS]
    gain     = np.mean(pers_f1s) - np.mean(base_f1s)

    try:
        stat, p = wilcoxon(pers_f1s, base_f1s, zero_method="zsplit")
        sig = "YES *" if p < alpha_bonf else ("yes (uncorr)" if p < alpha else "no")
    except Exception:
        p   = float("nan")
        sig = "n/a (no variance)"

    print(fmt.format(
        LABEL_DISPLAY.get(label, label),
        f"{np.mean(base_f1s):.3f}",
        f"{np.mean(pers_f1s):.3f}",
        f"{gain:+.3f}",
        f"{p:.4f}" if not np.isnan(p) else "  n/a",
        sig
    ))

print("-" * 75)
print("  * Significant after Bonferroni correction")
print("  yes (uncorr) = significant without but not after Bonferroni correction")

---
## STAT CELL D · 95% Confidence Intervals — Baseline and Personalised Accuracy
Reports Wilson CIs for each participant's baseline and personalised accuracy.
These can be added directly to your results table in the paper.

In [ ]:
from statsmodels.stats.proportion import proportion_confint
import numpy as np

# Test phase has 70 prompts (10 per category x 7 categories)
# Adaptation phase has 140 prompts
N_TEST  = 70
N_TRAIN = 140

print("=" * 80)
print("95% Wilson Confidence Intervals — Baseline and Personalised Accuracy")
print("=" * 80)

fmt = "{:<12} {:>12} {:>20} {:>12} {:>20}"
print(fmt.format("Participant", "Base Acc", "Base 95% CI", "Pers Acc", "Pers 95% CI"))
print("-" * 80)

for u in USERS:
    # Baseline: from training phase
    b_correct = round(u["baseline_acc"] * N_TRAIN)
    b_lo, b_hi = proportion_confint(b_correct, N_TRAIN, method="wilson")

    # Personalised: from test phase
    p_correct = round(u["test_acc"] * N_TEST)
    p_lo, p_hi = proportion_confint(p_correct, N_TEST, method="wilson")

    print(fmt.format(
        u["name"],
        f"{u['baseline_acc']:.3f}",
        f"[{b_lo:.3f}, {b_hi:.3f}]",
        f"{u['test_acc']:.3f}",
        f"[{p_lo:.3f}, {p_hi:.3f}]"
    ))

print("-" * 80)
mean_base = np.mean([u["baseline_acc"] for u in USERS])
mean_pers = np.mean([u["test_acc"]     for u in USERS])
print(fmt.format("Mean", f"{mean_base:.3f}", "", f"{mean_pers:.3f}", ""))
print("=" * 80)

---
## STAT CELL E · Update Count vs Personalisation Gain — Spearman Correlation
Formally quantifies the relationship between number of model updates
and accuracy gain shown in your scatter plot (Figure 6).

In [ ]:
from scipy.stats import spearmanr, pearsonr
import numpy as np

updates = np.array([u["updates"] for u in USERS])
gains   = np.array([u["test_acc"] - u["baseline_acc"] for u in USERS])

rho, p_spear  = spearmanr(updates, gains)
r,   p_pears  = pearsonr(updates,  gains)

print("=" * 60)
print("Correlation: Model Updates vs Personalisation Gain")
print("=" * 60)
print(f"  Updates  : {list(updates)}")
print(f"  Gains    : {[round(g,4) for g in gains]}")
print("")
print(f"  Spearman rho = {rho:.4f},  p = {p_spear:.4f}")
print(f"  Pearson  r   = {r:.4f},    p = {p_pears:.4f}")
print("")
sig_s = 'SIGNIFICANT' if p_spear < 0.05 else 'not significant'
sig_p = 'SIGNIFICANT' if p_pears < 0.05 else 'not significant'
print(f"  Spearman -> {sig_s} at alpha = 0.05")
print(f"  Pearson  -> {sig_p} at alpha = 0.05")
print("")
print("  NOTE: n=5 is small; use Spearman as primary (non-parametric).")
print("=" * 60)

## Cell 12 · Fig 9 — Confidence Distribution

Prediction confidence for correct vs incorrect predictions,
pooled across all users. Side-by-side: Phase 1 vs Phase 2.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, phase_key, title in zip(
    axes,
    ["train_df", "test_df"],
    ["Phase 1 — Adaptation (140 samples/user)",
     "Phase 2 — Held-out Test (70 samples/user)"],
):
    cor, inc = [], []
    for u in USERS:
        df = u[phase_key]
        cor.extend(df[df["correct"]]["confidence"].tolist())
        inc.extend(df[~df["correct"]]["confidence"].tolist())

    ax.hist(cor, bins=20, range=(0, 1), color="#5B9BD5", alpha=0.75,
            label=f"Correct  (n={len(cor)})", edgecolor="white")
    ax.hist(inc, bins=20, range=(0, 1), color="tomato",  alpha=0.75,
            label=f"Incorrect (n={len(inc)})", edgecolor="white")
    if cor:
        ax.axvline(np.mean(cor), color="navy", linestyle="--", linewidth=1.4,
                   label=f"μ correct = {np.mean(cor):.3f}")
    if inc:
        ax.axvline(np.mean(inc), color="darkred", linestyle="--", linewidth=1.4,
                   label=f"μ incorrect = {np.mean(inc):.3f}")
    ax.set_xlabel("Prediction Confidence")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle("Fig 9 — Confidence: Correct vs Incorrect  (all participants)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
_save(fig, "fig9_confidence_distribution.png")


## Cell 13 · Fig 10 — 6-Panel Summary Dashboard

All key results in one figure. Suitable as a supplementary figure.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
ax = axes.flatten()

# (a) Learning curves
for i, u in enumerate(USERS):
    ax[0].plot(range(1, len(u["train_df"]) + 1),
               u["train_df"]["running_acc"].values,
               color=USER_COLORS[i], linewidth=1.5, label=u["name"])
min_len  = min(len(u["train_df"]) for u in USERS)
mean_acc = np.array([u["train_df"]["running_acc"].values[:min_len]
                     for u in USERS]).mean(axis=0)
ax[0].plot(range(1, min_len + 1), mean_acc,
           "k--", linewidth=2.2, label="Mean", zorder=10)
ax[0].set_xlim(1, 140); ax[0].set_ylim(0, 1.05)
ax[0].set_xlabel("Sample #"); ax[0].set_ylabel("Accuracy")
ax[0].set_title("(a) Running Accuracy — Phase 1", fontweight="bold")
ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)

# (b) Baseline vs personalised
names_d = [u["name"]         for u in USERS]
base_d  = [u["baseline_acc"] for u in USERS]
pers_d  = [u["test_acc"]     for u in USERS]
x2 = np.arange(len(names_d)); w2 = 0.35
ax[1].bar(x2 - w2/2, base_d, w2, color="#5B9BD5", label="Baseline", edgecolor="white")
ax[1].bar(x2 + w2/2, pers_d, w2, color="#ED7D31", label="Personalised", edgecolor="white")
ax[1].set_xticks(x2); ax[1].set_xticklabels(names_d, fontsize=8)
ax[1].set_ylim(0, 1.2); ax[1].set_ylabel("Accuracy")
ax[1].set_title("(b) Baseline vs Personalised", fontweight="bold")
ax[1].legend(fontsize=8); ax[1].grid(axis="y", alpha=0.3)

# (c) Per-class F1
mb = [np.mean([u["f1_base"][l] for u in USERS]) for l in LABEL_ORDER]
mp = [np.mean([u["f1_pers"][l] for u in USERS]) for l in LABEL_ORDER]
x3 = np.arange(len(LABEL_ORDER)); w3 = 0.35
ax[2].bar(x3 - w3/2, mb, w3, color="#5B9BD5", label="Baseline", edgecolor="white")
ax[2].bar(x3 + w3/2, mp, w3, color="#ED7D31", label="Personalised", edgecolor="white")
ax[2].set_xticks(x3)
ax[2].set_xticklabels([LABEL_DISPLAY[l] for l in LABEL_ORDER],
                      rotation=35, ha="right", fontsize=7)
ax[2].set_ylim(0, 1.2); ax[2].set_ylabel("F1-Score")
ax[2].set_title("(c) Per-Class F1 (Mean)", fontweight="bold")
ax[2].legend(fontsize=8); ax[2].grid(axis="y", alpha=0.3)

# (d) Aggregated confusion matrix
all_t, all_p = [], []
for u in USERS:
    all_t.extend(u["test_df"]["true_label"].tolist())
    all_p.extend(u["test_df"]["pred_label"].tolist())
present4 = [l for l in LABEL_ORDER if l in set(all_t)]
cm4 = confusion_matrix(all_t, all_p, labels=present4)
ConfusionMatrixDisplay(
    cm4, display_labels=[LABEL_DISPLAY[l] for l in present4]
).plot(ax=ax[3], colorbar=False, cmap="Blues", xticks_rotation=40)
ax[3].set_title("(d) Aggregated Confusion Matrix", fontweight="bold")
for txt in ax[3].texts: txt.set_fontsize(7)

# (e) Updates per user
upd = [u["updates"] for u in USERS]
bars5 = ax[4].bar(names_d, upd,
                  color=[USER_COLORS[i] for i in range(len(USERS))],
                  edgecolor="white", width=0.5)
for bar, val in zip(bars5, upd):
    ax[4].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
               str(val), ha="center", fontsize=9, fontweight="bold")
ax[4].axhline(np.mean(upd), color="black", linestyle="--",
              linewidth=1.2, label=f"Mean = {np.mean(upd):.1f}")
ax[4].set_ylabel("No. of Model Updates")
ax[4].set_title("(e) Model Updates per Participant", fontweight="bold")
ax[4].legend(fontsize=8); ax[4].grid(axis="y", alpha=0.3)

# (f) Confidence — test phase
cor6, inc6 = [], []
for u in USERS:
    df = u["test_df"]
    cor6.extend(df[df["correct"]]["confidence"].tolist())
    inc6.extend(df[~df["correct"]]["confidence"].tolist())
ax[5].hist(cor6, bins=15, range=(0,1), color="#5B9BD5", alpha=0.75,
           label=f"Correct (n={len(cor6)})", edgecolor="white")
ax[5].hist(inc6, bins=15, range=(0,1), color="tomato", alpha=0.75,
           label=f"Incorrect (n={len(inc6)})", edgecolor="white")
if cor6: ax[5].axvline(np.mean(cor6), color="navy",    linestyle="--", linewidth=1.2)
if inc6: ax[5].axvline(np.mean(inc6), color="darkred", linestyle="--", linewidth=1.2)
ax[5].set_xlabel("Confidence"); ax[5].set_ylabel("Count")
ax[5].set_title("(f) Confidence (Test Phase)", fontweight="bold")
ax[5].legend(fontsize=8); ax[5].grid(alpha=0.3)

plt.suptitle("Fig 10 — Voice-to-Grasp: Summary Dashboard",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
_save(fig, "fig10_summary_dashboard.png")
